# Data Description
- Author: 詹明翰、謝子詮、曾翊涵（共同作者不等於已完成個別工作）
- Date first created: 20261127
- Background: 固定456里單一都市計畫快照補充資料，五類可靠工作分類面積／observed_share、條件式上下界及寬度、其他已分類面積、完整不確定聯集與品質。不是完整分區比例、實際土地使用或主要事故分群輸入。 本文件獨立讀正式CSV／schema，不重跑GIS或舊Notebook，未建立正式Finding。
- Unit of observation: 固定研究行政里，456列，village_id唯一；沒有year或S0–S3重複。
- Sample period: 單一都市計畫固定快照，沿Unit1.4內容標示2025-01、資源2026-04-09及實際DBF日期分開記錄；均不等於全部個案法定生效日，不宣稱2024／2025兩套觀測或事故前環境。
- Known issues: 面層原CRS缺PRJ，僅記憶體EPSG:3826工作假設B級，非A級認證。原有效面不修復，105無效面算法一致候選採linework；38方法敏感以三法面候選聯集保守扣除，buffer0只作比較；非面構件另記。固定原里界不修補，正式分母沿Unit2.2。衍生分區overlay採10^-4m一致運算網格控制GEOS數值殘留，原圖資／原里界座標不變，這不是實際位置精度認證；逐里面積關係容忍0.1m²、衍生分類交集檢查容忍0.01m²；全市聚合面積關係另容忍0.1m²（小於全市面積的4×10^-10），不clip比例。陽明山公園仍unclassified，不按景觀猜公園。五類不正規化，特殊混合等已知非五主類不當未知。不確定來源可重疊，總量用union。已辨識零不代表確定不存在；處理失敗保留ID／NA。L=observed，U=min(1,L+Q/A)，是依工作分類正確及敏感範圍包絡足夠等假設的條件範圍，不是信賴區間、法定真值或土地使用真值界，各上界不能同時實現，不以中點補值。品質門檻僅候選，不排除其他資料集里。城市union、逐里面積和加權、里算術平均及中位數分開；原里界微量重疊保留。D改良版／Prediction暫緩，無正式EDA／模型／推論。成員交付確認與DA／PM驗收待完成。
- Definition for each variable: 下方完整127欄字典逐一列dtype／定義／單位／來源／缺失規則。面積m²／km²、完整固定分母、observed／L／U／width及零／未知／失敗語義均保留。
- Data path: `data/processed/taipei_dataset_b2.csv`
- Source data: 原都市計畫六配套（點層不算面積）、固定里界五配套、正式里主表、v1 crosswalk及既有證據，25項來源／helper固定SHA見B2_INPUTS；12項A／B1／C僅交付鍵檢查，不作分區特徵值輸入。
- Processing code: [處理Notebook](../code/prepare_taipei_dataset_b2.ipynb)；[helper](../code/unit25_zoning_support.py)。Producer SHA `d3666314a709587732dc23905837d3c62014cd3cc1b142c7221037dc21d1b00e`。
- Role: supplementary_external_zoning
- Verification status: 正式資料456×127、770,517 bytes，先經處理NotebookGIS／分類／面積與精確讀回核對；下方Assertions保存本文件乾淨kernel實際結果。技術通過不代表人工或官方核定。
- SHA-256: `1af7ce133e62a1188e8755a1e4e7bc55db8b89d639f089f1c75319c8a9ee0269`
- Schema: `data/processed/taipei_dataset_b2_schema.json`；SHA `d5857362444df6bea1f606330d0363d792cc1a7f56812c9bb4917ce1b5e3a102`。
- Downstream analysis input: `data/processed/taipei_dataset_b2.csv`（與Data path相同；crosswalk為處理支援而非研究觀察樣本）

Primary Owner：謝子詮（DE）；Collaborator / Analytical Reviewer：詹明翰（DA）；Project Reviewer：曾翊涵（PM）。Deliverables：本資料／schema／獨立說明及共用處理。Actual Contribution：Codex協助，保留共同作者，不推定成員實際交付／驗收完成。

數值語義：核心observed與所有條件式L／U嚴格0–1；十個品質來源比例可有浮點微量超界，容忍為每里0.1m²除以完整里面積。原比例不clip，quality_share_roundoff_flag及quality_share_roundoff_max_excess_m2記錄超界與超額面積；不把非分區邊界當有效分區分類。


## Packages、來源與正式版本

In [1]:
from pathlib import Path
import sys,json,hashlib,platform
import pandas as pd
import numpy as np
from IPython.display import display,Markdown
for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents):
    if (p/'AGENTS.md').is_file() and (p/'templates').is_dir():
        PROJECT_ROOT=p;break
else:raise FileNotFoundError('請從研究工作包內執行')
assert Path(sys.executable).resolve()==(PROJECT_ROOT/'.venv/Scripts/python.exe').resolve()
def sha(path):
    h=hashlib.sha256()
    with path.open('rb') as stream:
        for part in iter(lambda:stream.read(4*1024*1024),b''):h.update(part)
    return h.hexdigest()
assert sha(PROJECT_ROOT/'data/code/unit25_zoning_support.py')=='a5d1a8bbc8b0473921340b01af09506902123030502a61035def77c83af7b2a4'
sys.path.insert(0,str(PROJECT_ROOT/'data/code'))
import unit25_zoning_support as u
B2_INPUTS={'data/raw/細計-面.shp': '67d2ecc469383459e43f2609cf384a4129e09b27c590cbbb82dfa750eba758ee', 'data/raw/細計-面.shx': '277e5f623c5c81865aa7288fac979a279b340899a4b3bc4335638f797fc5fd69', 'data/raw/細計-面.dbf': '29b0484feef66c6c8417821ada8a28ed1edc0efcdffb492476fe7ada8739e1d4', 'data/raw/細計-文.shp': 'fa1767fc78c69b7683ec53e0fab8937bac6c33868bedee3c3dcc2ed38dcab731', 'data/raw/細計-文.shx': '32d642964d494f49d08830f9c12857f9f4995b56eebf09ef2be994034b864760', 'data/raw/細計-文.dbf': 'eba92881fc11e574f24024380c946d73cf213e86040663c4070d15b1d552a7be', 'data/raw/G97_A_CAVLGE_P.shp': 'f7fb4813a2283198ca564f5686823c454c5fd832d2f6981a2ee88a94719be3cc', 'data/raw/G97_A_CAVLGE_P.shx': 'ab7bc5e63229f676d3580b68e98db59b91a2d327c74ca23b0000ca1c568d414e', 'data/raw/G97_A_CAVLGE_P.dbf': '2f43f92e84279a45b2512cf453e9b72f706d1ad1402665c946b1a354f75d8577', 'data/raw/G97_A_CAVLGE_P.prj': '8dac7d82ea889063f9eaa7df408fcaeb7307d32831ea973b6534d0139a4523c5', 'data/raw/G97_A_CAVLGE_P.cpg': '3ad3031f5503a4404af825262ee8232cc04d4ea6683d42c5dd0a2f2a27ac9824', 'data/processed/taipei_village_base.csv': 'db0a225df3568ddb2a47eab1e1d28843e1594dec37956c57eea926e444968209', 'data/processed/taipei_village_base_schema.json': 'a7e5e3b4d77ab506736ecc940e26a87df6cace3a738cbdaeef1e835e6bed7070', 'data/processed/taipei_village_base_documentation.ipynb': 'a843d587788b4ea6a943cc05d9413908cf8b66b2f73f9fdbec01e2ab52edd535', 'data/processed/urban_zoning_crosswalk.csv': '4326d92d1567167ca67c13dc1c083426bf6d4d789b88ed5da1feec422306383f', 'data/processed/urban_zoning_crosswalk_documentation.ipynb': 'b931add933d943890b9ec8fbed463836d28838632cec3a5f0636ff0f6b9fa36d', 'data/raw/taipei_urban_planning_raw_documentation.ipynb': 'bb3dd6bca248ce866b30766762d100cd926e7cbed30f0165381f531709b8aa8f', 'data/raw/unit14_urban_planning_evidence.zip': '695e15f7230c076b2b827f72b4337273ff795f6fc7e9c52680641c972dd72730', 'data/raw/unit14_urban_planning_evidence_documentation.ipynb': 'ec2e9b002441d10b26b426f6d43cf6c5dd9c598ca058cda81a2f9f058e02dc55', 'data/raw/unit15_zoning_classification_evidence.zip': '82effe1535fc7577d73accf81ee0663daa78944a11517b9e50c7bc2e197ba9b9', 'data/raw/unit15_zoning_classification_evidence_documentation.ipynb': '15f164fd91f2b6459c56e789001bc928e8dd96f01e14e2ff4c311e758ce14c35', 'data/code/taipei_urban_planning_crs_validation.ipynb': 'cf9cd6a6d46a943fab2f148db114d1ad49a605a664a2ee6109761df8e588b839', 'data/code/taipei_urban_planning_geometry_validation.ipynb': 'b5c8b2358dc9774c744639c94d85961e44ec8230f223d63387cd6846605e04b7', 'data/code/taipei_data_integration_readiness.ipynb': '6048ff52d7653c9ec9d4b32cf2b78da2fb1b22dc0db0164cc48c4d2d98dd338b', 'data/code/unit25_zoning_support.py': 'a5d1a8bbc8b0473921340b01af09506902123030502a61035def77c83af7b2a4'}
HANDOFF_ONLY_INPUTS={'data/processed/taipei_dataset_a_annual.csv': '4e264176c970263777eb4c564f427e1f839a45b31552a37bc4bdbfbed6b09422', 'data/processed/taipei_dataset_a_annual_schema.json': '4d0c5b7b307b1cca4261a9595cd1ca894f549d88e06bed1b0b071dcf7c40fb21', 'data/processed/taipei_dataset_a_pooled.csv': '081c7dd42c94e16a5684f27edaebd310a9c556bae7460783dd87fde8d26ea46a', 'data/processed/taipei_dataset_a_pooled_schema.json': '7878146fac39d4bfabee965b6001d0cf5e101ca1d76778599fec4b480ad59b5a', 'data/processed/taipei_dataset_b1_annual.csv': 'd4ad76e82ad6f00d5b146c00c0e065d2095cd9fd36fe6bbbad5feb40705ae795', 'data/processed/taipei_dataset_b1_annual_schema.json': '89bb5ace9a7abf9bcedcc1ff0d6b2f91d45578f7c6cfe20a685bd6802d8fa7ca', 'data/processed/taipei_dataset_b1_pooled.csv': '3b02fb7b338b59f2552972ccb98efa0d723e525826b301cdda021efa9ddd4e50', 'data/processed/taipei_dataset_b1_pooled_schema.json': '63f46e1ae0cba9c19549ee5f8b9ad39084a22f5d2d738b62567b1b3f5e86f340', 'data/processed/taipei_dataset_c_annual.csv': '9ab1970ed0f790ebdf31e1726094493305da00ac4efaf84b8a554f7debfa560e', 'data/processed/taipei_dataset_c_annual_schema.json': '4a8151a9a5dcd716cc03312b303c4455cb47210192c43b86cf3a0e06d268cb6f', 'data/processed/taipei_dataset_c_pooled.csv': 'caceea61ff906b3e2f0b4796a31af3373a929307a0c7f7094fb9373794f76bc6', 'data/processed/taipei_dataset_c_pooled_schema.json': '44baac19975650d792e23e81c5463b84e00f299c9c606ef0f95abf8b3756d3ab'}
u.verify(PROJECT_ROOT,B2_INPUTS);u.verify(PROJECT_ROOT,HANDOFF_ONLY_INPUTS)
print('Python',platform.python_version(),'B2來源／helper hash',len(B2_INPUTS),'僅交付鍵來源hash',len(HANDOFF_ONLY_INPUTS))

EXPECTED_OUTPUTS={'data/processed/taipei_dataset_b2.csv': '1af7ce133e62a1188e8755a1e4e7bc55db8b89d639f089f1c75319c8a9ee0269', 'data/processed/taipei_dataset_b2_schema.json': 'd5857362444df6bea1f606330d0363d792cc1a7f56812c9bb4917ce1b5e3a102', 'data/processed/urban_zoning_crosswalk_v2.csv': '3af29de364a10cd135252b9851ce6bbb74586ab47d5c5d68ade036e4a2960583', 'data/processed/urban_zoning_crosswalk_v2_schema.json': '5925bf3b77fd129bfd1e7eb4aaf765596b4d26fe0228b1c34f3a9935f9711a5f'}
assert sha(PROJECT_ROOT/'data/code/prepare_taipei_dataset_b2.ipynb')=='d3666314a709587732dc23905837d3c62014cd3cc1b142c7221037dc21d1b00e'
for rel,h in EXPECTED_OUTPUTS.items():assert sha(PROJECT_ROOT/rel)==h,('OUTPUT HASH MISMATCH',rel)
print('四正式輸出與producer版本核對通過；只讀saved files')


Python 3.12.2 B2來源／helper hash 25 僅交付鍵來源hash 12
四正式輸出與producer版本核對通過；只讀saved files


## Change for Your Own Data

精確路徑與hash先核對，再按schema讀取，保留原空字串、nullable dtype及`__MISSING__`，不使用預設NA推論或猜版本。

In [2]:
saved={};contracts={}
for name in [u.B2,u.CW]:
    contract=json.loads((PROJECT_ROOT/f'data/processed/{name}_schema.json').read_text(encoding='utf-8'))
    assert contract['source_inputs']==B2_INPUTS and contract['handoff_only_inputs']==HANDOFF_ONLY_INPUTS
    saved[name]=u.read_csv(PROJECT_ROOT/f'data/processed/{name}.csv',contract);contracts[name]=contract
base_contract=json.loads((PROJECT_ROOT/'data/processed/taipei_village_base_schema.json').read_text(encoding='utf-8'))
base=u.read_csv(PROJECT_ROOT/'data/processed/taipei_village_base.csv',base_contract)
old=pd.read_csv(PROJECT_ROOT/'data/processed/urban_zoning_crosswalk.csv',encoding='utf-8-sig',dtype='string',keep_default_na=False)
u.validate_b2(saved[u.B2],base);u.validate_crosswalk(saved[u.CW],old)
for col in old:
    if col=='research_zoning_category':continue
    pd.testing.assert_series_equal(saved[u.CW][col].astype('string'),old[col].astype('string'),check_names=False,check_exact=True)
STEM='taipei_dataset_b2'
frame=saved[STEM];contract=contracts[STEM]
assert frame.shape==(456,127)
print('Rows',len(frame),'Columns',frame.shape[1],'primary key',contract['primary_key'])

Rows 456 Columns 127 primary key ['village_id']


## Definition for each variable

In [3]:
dictionary=pd.DataFrame(contract['columns'])
u.show('Definition for each variable：完整資料字典',dictionary,block=16)
assert list(frame)==dictionary['name'].tolist()
assert [str(frame[c].dtype) for c in frame]==dictionary['dtype'].tolist()

**Definition for each variable：完整資料字典**

,name,dtype,definition,unit,source,missing_rule
0,village_id,string,Unit2.2固定里主表之village_id,not applicable,Unit2.2 village base,不可缺失，糖廍正常納入
1,boundary_code,string,Unit2.2固定里主表之boundary_code,not applicable,Unit2.2 village base,不可缺失，糖廍正常納入
2,district_name,string,Unit2.2固定里主表之district_name,not applicable,Unit2.2 village base,不可缺失，糖廍正常納入
3,standard_village_name,string,Unit2.2固定里主表之standard_village_name,not applicable,Unit2.2 village base,不可缺失，糖廍正常納入
4,verification_status,string,Unit2.2固定里主表之verification_status,not applicable,Unit2.2 village base,不可缺失，糖廍正常納入
5,geometry_quality_flag,string,Unit2.2固定里主表之geometry_quality_flag,not applicable,Unit2.2 village base,不可缺失，糖廍正常納入
6,fixed_village_area_m2,Float64,完整原固定里幾何面積，所有比例的共同分母,m²,Unit2.2 village_area_m2,正值不可缺失
7,fixed_village_area_km2,Float64,原固定里面積，無修補／填洞／去重疊,km²,Unit2.2 village_area_km2,正值不可缺失
8,crosswalk_version,string,指定兩鍵修訂，非法定分區變更,not applicable,Unit1.4–1.6 evidence / Unit2.5 working specification,原時間語義限制保留
9,working_crs,string,記憶體工作CRS，不補原PRJ,not applicable,Unit1.4–1.6 evidence / Unit2.5 working specification,原時間語義限制保留


,name,dtype,definition,unit,source,missing_rule
16,year_specific_observation,boolean,單一固定快照，無獨立2024／2025分區觀測,not applicable,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
17,candidate_zoning_area_m2,Float64,candidate_zoning幾何與原里實際交集面積；來源集合可能相互重疊,m²,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
18,candidate_zoning_area_km2,Float64,candidate_zoning交集面積,km²,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
19,candidate_zoning_coverage_share,Float64,candidate_zoning交集／完整里面積，非各來源可相加的互斥分類,fraction,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
20,classified_candidate_area_m2,Float64,classified_candidate幾何與原里實際交集面積；來源集合可能相互重疊,m²,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
21,classified_candidate_area_km2,Float64,classified_candidate交集面積,km²,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
22,classified_candidate_coverage_share,Float64,classified_candidate交集／完整里面積，非各來源可相加的互斥分類,fraction,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
23,identified_zoning_area_m2,Float64,identified_zoning幾何與原里實際交集面積；來源集合可能相互重疊,m²,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
24,identified_zoning_area_km2,Float64,identified_zoning交集面積,km²,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
25,identified_zoning_coverage_share,Float64,identified_zoning交集／完整里面積，非各來源可相加的互斥分類,fraction,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零


,name,dtype,definition,unit,source,missing_rule
32,cross_class_overlap_area_m2,Float64,cross_class_overlap幾何與原里實際交集面積；來源集合可能相互重疊,m²,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
33,cross_class_overlap_area_km2,Float64,cross_class_overlap交集面積,km²,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
34,cross_class_overlap_share,Float64,cross_class_overlap交集／完整里面積，非各來源可相加的互斥分類,fraction,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
35,classified_unclassified_overlap_area_m2,Float64,classified_unclassified_overlap幾何與原里實際交集面積；來源集合可能相互重疊,m²,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
36,classified_unclassified_overlap_area_km2,Float64,classified_unclassified_overlap交集面積,km²,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
37,classified_unclassified_overlap_share,Float64,classified_unclassified_overlap交集／完整里面積，非各來源可相加的互斥分類,fraction,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
38,geometry_sensitive_area_m2,Float64,geometry_sensitive幾何與原里實際交集面積；來源集合可能相互重疊,m²,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
39,geometry_sensitive_area_km2,Float64,geometry_sensitive交集面積,km²,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
40,geometry_sensitive_coverage_share,Float64,geometry_sensitive交集／完整里面積，非各來源可相加的互斥分類,fraction,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
41,uncovered_area_m2,Float64,uncovered幾何與原里實際交集面積；來源集合可能相互重疊,m²,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零


,name,dtype,definition,unit,source,missing_rule
48,quality_share_roundoff_max_excess_m2,Float64,十個品質來源中超過完整里面積之最大正超額，零代表無超額；只作數值稽核,m²,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
49,agriculture_conservation_identified_area_m2,Float64,agriculture_conservation經同類union及方案B扣除不確定範圍後已辨識面積,m²,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
50,agriculture_conservation_identified_area_km2,Float64,agriculture_conservation可靠工作分類面積，不等同實際土地使用,km²,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
51,agriculture_conservation_identified_share,Float64,agriculture_conservation已辨識面積／完整里面積，不以已辨識總面積為分母,fraction,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
52,commercial_identified_area_m2,Float64,commercial經同類union及方案B扣除不確定範圍後已辨識面積,m²,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
53,commercial_identified_area_km2,Float64,commercial可靠工作分類面積，不等同實際土地使用,km²,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
54,commercial_observed_share,Float64,commercial已辨識面積／完整里面積，不以已辨識總面積為分母,fraction,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
55,educational_identified_area_m2,Float64,educational經同類union及方案B扣除不確定範圍後已辨識面積,m²,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
56,educational_identified_area_km2,Float64,educational可靠工作分類面積，不等同實際土地使用,km²,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
57,educational_observed_share,Float64,educational已辨識面積／完整里面積，不以已辨識總面積為分母,fraction,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零


,name,dtype,definition,unit,source,missing_rule
64,other_classified_identified_area_m2,Float64,other_classified經同類union及方案B扣除不確定範圍後已辨識面積,m²,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
65,other_classified_identified_area_km2,Float64,other_classified可靠工作分類面積，不等同實際土地使用,km²,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
66,other_classified_identified_share,Float64,other_classified已辨識面積／完整里面積，不以已辨識總面積為分母,fraction,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
67,other_public_facilities_identified_area_m2,Float64,other_public_facilities經同類union及方案B扣除不確定範圍後已辨識面積,m²,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
68,other_public_facilities_identified_area_km2,Float64,other_public_facilities可靠工作分類面積，不等同實際土地使用,km²,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
69,other_public_facilities_identified_share,Float64,other_public_facilities已辨識面積／完整里面積，不以已辨識總面積為分母,fraction,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
70,park_green_identified_area_m2,Float64,park_green經同類union及方案B扣除不確定範圍後已辨識面積,m²,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
71,park_green_identified_area_km2,Float64,park_green可靠工作分類面積，不等同實際土地使用,km²,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
72,park_green_observed_share,Float64,park_green已辨識面積／完整里面積，不以已辨識總面積為分母,fraction,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
73,residential_identified_area_m2,Float64,residential經同類union及方案B扣除不確定範圍後已辨識面積,m²,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零


,name,dtype,definition,unit,source,missing_rule
80,water_identified_area_km2,Float64,water可靠工作分類面積，不等同實際土地使用,km²,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
81,water_identified_share,Float64,water已辨識面積／完整里面積，不以已辨識總面積為分母,fraction,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
82,core_five_identified_area_m2,Float64,五核心已辨識面積，不重新正規化,m²,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
83,core_five_identified_share,Float64,五核心已辨識面積，不重新正規化／完整里面積,fraction,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
84,other_noncore_identified_area_m2,Float64,其餘六個已分類類別合計，與other_classified單一類別分開；混合特殊是已知類別，不當未知,m²,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
85,other_noncore_identified_share,Float64,其餘六個已分類類別合計，與other_classified單一類別分開；混合特殊是已知類別，不當未知／完整里面積,fraction,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
86,residential_conditional_lower_bound,Float64,條件式L等於已辨識覆蓋；假定既有工作分類及包絡足夠，不是信賴區間,fraction,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
87,residential_conditional_upper_bound,Float64,"條件式U=min(1,L+不確定聯集比例)，非官方真值認證且各類上界不能同時實現",fraction,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
88,residential_conditional_bound_width,Float64,U−L；不以中點補成最佳估計,fraction,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
89,residential_observed_zero_flag,boolean,已辨識面積為0，不等於已證實該分區不存在,not applicable,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零


,name,dtype,definition,unit,source,missing_rule
96,industrial_conditional_lower_bound,Float64,條件式L等於已辨識覆蓋；假定既有工作分類及包絡足夠，不是信賴區間,fraction,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
97,industrial_conditional_upper_bound,Float64,"條件式U=min(1,L+不確定聯集比例)，非官方真值認證且各類上界不能同時實現",fraction,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
98,industrial_conditional_bound_width,Float64,U−L；不以中點補成最佳估計,fraction,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
99,industrial_observed_zero_flag,boolean,已辨識面積為0，不等於已證實該分區不存在,not applicable,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
100,industrial_interpretation_state,string,區分已觀測零／部分覆蓋／處理失敗，不宣稱實際分區確定零,not applicable,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
101,educational_conditional_lower_bound,Float64,條件式L等於已辨識覆蓋；假定既有工作分類及包絡足夠，不是信賴區間,fraction,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
102,educational_conditional_upper_bound,Float64,"條件式U=min(1,L+不確定聯集比例)，非官方真值認證且各類上界不能同時實現",fraction,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
103,educational_conditional_bound_width,Float64,U−L；不以中點補成最佳估計,fraction,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
104,educational_observed_zero_flag,boolean,已辨識面積為0，不等於已證實該分區不存在,not applicable,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
105,educational_interpretation_state,string,區分已觀測零／部分覆蓋／處理失敗，不宣稱實際分區確定零,not applicable,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零


,name,dtype,definition,unit,source,missing_rule
112,classified_sum_residual_m2,Float64,各類已辨識面積和−已辨識聯集I,m²,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
113,candidate_uncovered_residual_m2,Float64,候選覆蓋＋未覆蓋−完整里面積,m²,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
114,identified_uncertain_overlap_m2,Float64,I與Q實際交集之面積部分；線／點不算面積，只容許已明示浮點誤差,m²,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
115,coverage_ge_50,boolean,可靠辨識覆蓋≥50%候選旗標，不是正式排除政策,not applicable,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
116,coverage_ge_70,boolean,可靠辨識覆蓋≥70%候選旗標，不是正式排除政策,not applicable,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
117,coverage_ge_80,boolean,可靠辨識覆蓋≥80%候選旗標，不是正式排除政策,not applicable,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
118,coverage_ge_90,boolean,可靠辨識覆蓋≥90%候選旗標，不是正式排除政策,not applicable,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
119,coverage_ge_95,boolean,可靠辨識覆蓋≥95%候選旗標，不是正式排除政策,not applicable,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
120,coverage_ge_99,boolean,可靠辨識覆蓋≥99%候選旗標，不是正式排除政策,not applicable,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零
121,invalid_source_reference_intersection_count,Int64,原無效143筆linework面參考與里正面積相交數；不是官方認證修復,source feature count,Unit2.5 Scheme B derived geometry,處理失敗保留NA，不補零


## Summary

### Sample

In [4]:
u.show('實際前10列主要來源／地理欄',frame.iloc[:10,:18])
u.show('前10列五observed_share與不確定品質',frame.iloc[:10][['village_id']+[c+'_observed_share' for c in u.CORE]+['identified_zoning_coverage_share','uncertain_union_share','processing_status']])

**實際前10列主要來源／地理欄**

,village_id,boundary_code,district_name,standard_village_name,verification_status,geometry_quality_flag,fixed_village_area_m2,fixed_village_area_km2,crosswalk_version,working_crs,crs_evidence_grade,snapshot_source_description,snapshot_resource_version,analysis_role,source_dbf_header_date,legal_snapshot_effective_date,year_specific_observation,candidate_zoning_area_m2
0,6300100002,6300100002,松山區,莊敬里,verified_exact_match,valid_original,1281723.117747,1.281723,v2_unit25,EPSG:3826,B,2025-01,2026-04-09,supplementary_external_zoning,2026-04-09,<NA>,False,1141549.81792
1,6300100003,6300100003,松山區,東榮里,verified_exact_match,valid_original,259347.14737,0.259347,v2_unit25,EPSG:3826,B,2025-01,2026-04-09,supplementary_external_zoning,2026-04-09,<NA>,False,207080.369459
2,6300100004,6300100004,松山區,三民里,verified_exact_match,valid_original_overlap,134542.285747,0.134542,v2_unit25,EPSG:3826,B,2025-01,2026-04-09,supplementary_external_zoning,2026-04-09,<NA>,False,101107.718184
3,6300100005,6300100005,松山區,新益里,verified_exact_match,valid_original,115137.5782,0.115138,v2_unit25,EPSG:3826,B,2025-01,2026-04-09,supplementary_external_zoning,2026-04-09,<NA>,False,77812.394531
4,6300100006,6300100006,松山區,富錦里,verified_exact_match,valid_original,99809.362025,0.099809,v2_unit25,EPSG:3826,B,2025-01,2026-04-09,supplementary_external_zoning,2026-04-09,<NA>,False,76447.728761
5,6300100007,6300100007,松山區,新東里,verified_exact_match,valid_original_overlap,351945.654552,0.351946,v2_unit25,EPSG:3826,B,2025-01,2026-04-09,supplementary_external_zoning,2026-04-09,<NA>,False,284114.214174
6,6300100008,6300100008,松山區,富泰里,verified_exact_match,valid_original_overlap,57081.424959,0.057081,v2_unit25,EPSG:3826,B,2025-01,2026-04-09,supplementary_external_zoning,2026-04-09,<NA>,False,42875.270558
7,6300100009,6300100009,松山區,介壽里,verified_exact_match,valid_original,141203.777408,0.141204,v2_unit25,EPSG:3826,B,2025-01,2026-04-09,supplementary_external_zoning,2026-04-09,<NA>,False,114840.637198
8,6300100010,6300100010,松山區,精忠里,verified_exact_match,valid_original,2308512.493916,2.308512,v2_unit25,EPSG:3826,B,2025-01,2026-04-09,supplementary_external_zoning,2026-04-09,<NA>,False,2206548.391089
9,6300100011,6300100011,松山區,東光里,verified_exact_match,valid_original,187401.931968,0.187402,v2_unit25,EPSG:3826,B,2025-01,2026-04-09,supplementary_external_zoning,2026-04-09,<NA>,False,138472.95903


**前10列五observed_share與不確定品質**

,village_id,residential_observed_share,commercial_observed_share,industrial_observed_share,educational_observed_share,park_green_observed_share,identified_zoning_coverage_share,uncertain_union_share,processing_status
0,6300100002,0.045325,0.0,0.0,0.010506,0.014422,0.89063,0.10937,success_retained_uncertainty
1,6300100003,0.489969,0.0,0.0,0.0,0.18554,0.798468,0.201532,success_retained_uncertainty
2,6300100004,0.359684,0.158424,0.0,0.185302,0.011418,0.751494,0.248506,success_retained_uncertainty
3,6300100005,0.534422,0.0,0.0,0.00385,0.13755,0.675821,0.324179,success_retained_uncertainty
4,6300100006,0.507469,0.09827,0.0,0.0,0.094777,0.765937,0.234063,success_retained_uncertainty
5,6300100007,0.191207,0.005439,0.004461,0.049671,0.014709,0.807267,0.192733,success_retained_uncertainty
6,6300100008,0.440084,0.296403,0.0,0.0,0.014637,0.751125,0.248875,success_retained_uncertainty
7,6300100009,0.487029,0.0,0.0,0.10921,0.121811,0.813297,0.186703,success_retained_uncertainty
8,6300100010,0.022702,0.004504,0.0,0.005137,0.005563,0.955831,0.044169,success_retained_uncertainty
9,6300100011,0.443795,0.128815,0.0,0.13472,0.018919,0.738909,0.261091,success_retained_uncertainty


### Summary Stats

In [5]:
statistics=[]
for category in u.CORE:
    for suffix in ['_observed_share','_conditional_lower_bound','_conditional_upper_bound','_conditional_bound_width']:
        col=category+suffix;v=frame[col]
        statistics.append({'feature':col,'valid_N':int(v.count()),'NA_N':int(v.isna().sum()),'zero_N':int(v.eq(0).sum()),'min':float(v.min()),'q25':float(v.quantile(.25)),'median':float(v.median()),'q75':float(v.quantile(.75)),'max':float(v.max()),'mean':float(v.mean()),'std_ddof1':float(v.std())})
u.show('五核心及條件式L／U／寬度完整統計',pd.DataFrame(statistics))
print('逐里面積加總加權辨識比例',float(frame.identified_zoning_area_m2.sum()/frame.fixed_village_area_m2.sum()))
print('行政里辨識比例算術平均／中位數',float(frame.identified_zoning_coverage_share.mean()),float(frame.identified_zoning_coverage_share.median()))
print('全市union須由處理Notebook原幾何另算，不能把逐里和當全市union；正式source masks可由該流程重建')


**五核心及條件式L／U／寬度完整統計**

,feature,valid_N,NA_N,zero_N,min,q25,median,q75,max,mean,std_ddof1
0,residential_observed_share,456,0,41,0.000000,0.130775,0.303364,0.460769,0.928070,0.306901,0.209221
1,residential_conditional_lower_bound,456,0,41,0.000000,0.130775,0.303364,0.460769,0.928070,0.306901,0.209221
2,residential_conditional_upper_bound,456,0,0,0.019869,0.337115,0.566404,0.741079,1.000000,0.550954,0.244100
3,residential_conditional_bound_width,456,0,0,0.006028,0.156487,0.237706,0.306015,1.000000,0.244053,0.138626
4,commercial_observed_share,456,0,137,0.000000,0.000000,0.040421,0.142792,0.693845,0.100621,0.141306
5,commercial_conditional_lower_bound,456,0,137,0.000000,0.000000,0.040421,0.142792,0.693845,0.100621,0.141306
6,commercial_conditional_upper_bound,456,0,0,0.006028,0.179902,0.319617,0.466936,1.000000,0.344674,0.215241
7,commercial_conditional_bound_width,456,0,0,0.006028,0.156487,0.237706,0.306015,1.000000,0.244053,0.138626
8,industrial_observed_share,456,0,424,0.000000,0.000000,0.000000,0.000000,0.447537,0.011862,0.056238
9,industrial_conditional_lower_bound,456,0,424,0.000000,0.000000,0.000000,0.000000,0.447537,0.011862,0.056238


,feature,valid_N,NA_N,zero_N,min,q25,median,q75,max,mean,std_ddof1
18,park_green_conditional_upper_bound,456,0,0,0.009554,0.214629,0.289237,0.346713,1.0,0.297294,0.149384
19,park_green_conditional_bound_width,456,0,0,0.006028,0.156487,0.237706,0.306015,1.0,0.244053,0.138626


逐里面積加總加權辨識比例 0.6513777225684405
行政里辨識比例算術平均／中位數 0.7559468842332506 0.7622939966114667
全市union須由處理Notebook原幾何另算，不能把逐里和當全市union；正式source masks可由該流程重建


### 品質、修訂與解釋限制

In [6]:
u.show('六個候選覆蓋門檻，不決定正式排除',pd.DataFrame([{'threshold_percent':t,'villages':int(frame[f'coverage_ge_{t}'].sum()),'NA_flags':int(frame[f'coverage_ge_{t}'].isna().sum())} for t in u.THRESHOLDS]))
u.show('零／未知／處理失敗語義',pd.DataFrame([{'category':c,'observed_zero':int(frame[c+'_observed_zero_flag'].sum()),'NA_observed':int(frame[c+'_observed_share'].isna().sum()),'unknown_area_positive':int(frame.uncertain_union_share.gt(0).sum())} for c in u.CORE]))
print('品質比例微量超界里數／最大超額m²',int(frame.quality_share_roundoff_flag.sum()),float(frame.quality_share_roundoff_max_excess_m2.max()))
print('共同法定生效日未知里數',int(frame.legal_snapshot_effective_date.isna().sum()))
u.show('可靠辨識I為0但保留ID及不確定範圍，非處理失敗或確定無分區',frame.loc[frame.identified_zoning_area_m2.eq(0),['village_id','district_name','standard_village_name','identified_zoning_coverage_share','uncertain_union_share','processing_status']+[c+'_conditional_upper_bound' for c in u.CORE]])
d=frame.groupby('district_name').agg(villages=('village_id','size'),area_m2=('fixed_village_area_m2','sum'),identified_m2=('identified_zoning_area_m2','sum'),uncertain_m2=('uncertain_union_area_m2','sum'),unclassified_m2=('unclassified_area_m2','sum'),sensitive_m2=('geometry_sensitive_area_m2','sum'),uncovered_m2=('uncovered_area_m2','sum'))
d['identified_area_weighted_share']=d.identified_m2/d.area_m2
u.show('12區逐里面積加權品質，來源可能重疊不可直接相加',d.reset_index())
u.show('糖廍與士林公館完整保留',frame.loc[frame.village_id.isin(['6300700015','6301100044']),['village_id','standard_village_name','verification_status','identified_zoning_coverage_share','uncertain_union_share','processing_status']])


**六個候選覆蓋門檻，不決定正式排除**

,threshold_percent,villages,NA_flags
0,50,441,0
1,70,331,0
2,80,162,0
3,90,49,0
4,95,16,0
5,99,2,0


**零／未知／處理失敗語義**

,category,observed_zero,NA_observed,unknown_area_positive
0,residential,41,0,456
1,commercial,137,0,456
2,industrial,424,0,456
3,educational,219,0,456
4,park_green,82,0,456


品質比例微量超界里數／最大超額m² 185 0.00011237026774324477
共同法定生效日未知里數 456


**可靠辨識I為0但保留ID及不確定範圍，非處理失敗或確定無分區**

,village_id,district_name,standard_village_name,identified_zoning_coverage_share,uncertain_union_share,processing_status,residential_conditional_upper_bound,commercial_conditional_upper_bound,industrial_conditional_upper_bound,educational_conditional_upper_bound,park_green_conditional_upper_bound
455,6301200042,北投區,湖田里,0.0,1.0,success_retained_uncertainty,1.0,1.0,1.0,1.0,1.0


**12區逐里面積加權品質，來源可能重疊不可直接相加**

,district_name,villages,area_m2,identified_m2,uncertain_m2,unclassified_m2,sensitive_m2,uncovered_m2,identified_area_weighted_share
0,中山區,42,13918233.18613,9861519.233701,4056713.95179,0.0,1330223.605143,2726477.908976,0.708532
1,中正區,31,7541170.880451,5646374.297705,1894796.582746,0.0,10323.26353,1884473.289136,0.74874
2,信義區,41,11242119.269773,8706806.931467,2535312.338113,0.0,940032.825119,1595278.177682,0.774481
3,內湖區,39,31943100.977179,28178149.477744,3764951.499449,0.0,1067281.455769,2697636.163993,0.882136
4,北投區,42,57381714.454259,22346004.168337,35035710.283867,27469548.90284,4964220.320236,2601736.221666,0.389427
5,南港區,20,21953596.060832,20605881.586499,1347714.474403,0.0,123603.539075,1226215.897331,0.938611
6,士林區,51,61078025.482908,33058271.231132,28019754.240767,20455996.251975,4380916.188308,3182537.442475,0.541247
7,大同區,25,4768702.088781,3490507.86948,1278194.219301,0.0,203524.252129,1074668.853785,0.731962
8,大安區,53,11341386.541063,8566077.720909,2775308.820155,0.0,0.0,2775306.821991,0.755294
9,文山區,43,31250331.293149,21264003.145756,9986328.143457,0.0,7807568.521818,2178585.029116,0.680441


**糖廍與士林公館完整保留**

,village_id,standard_village_name,verification_status,identified_zoning_coverage_share,uncertain_union_share,processing_status
239,6300700015,糖廍里,provisional_match,0.728568,0.271432,success_retained_uncertainty
406,6301100044,公館里,verified_exact_match,0.965554,0.034446,success_retained_uncertainty


### 使用範例與技術交付

In [7]:
handoff=u.handoff_check(PROJECT_ROOT,frame,HANDOFF_ONLY_INPUTS)
u.show('A／B1／C以village_id many-to-one連同一快照',pd.DataFrame(handoff))
print('品質不足或NA不刪其他資料；不產出整合表，不把快照當年度分區變數或事故前因果環境')

**A／B1／C以village_id many-to-one連同一快照**

,dataset,rows_before,rows_after,unmatched,merge,B2_rows,B2_NA_rows_retained,feature_values_used_as_B2_input,missing_sentinel
0,taipei_dataset_a_annual,3648,3648,0,many_to_one,456,0,False,__MISSING__
1,taipei_dataset_a_pooled,1824,1824,0,many_to_one,456,0,False,__MISSING__
2,taipei_dataset_b1_annual,912,912,0,many_to_one,456,0,False,__MISSING__
3,taipei_dataset_b1_pooled,456,456,0,many_to_one,456,0,False,__MISSING__
4,taipei_dataset_c_annual,3648,3648,0,many_to_one,456,0,False,__MISSING__
5,taipei_dataset_c_pooled,1824,1824,0,many_to_one,456,0,False,__MISSING__


品質不足或NA不刪其他資料；不產出整合表，不把快照當年度分區變數或事故前因果環境


### Assertions

獨立saved-file驗證不重跑GIS；技術通過仍待人工驗收。

In [8]:
u.validate_b2(saved[u.B2],base);u.validate_crosswalk(saved[u.CW],old)
u.verify(PROJECT_ROOT,B2_INPUTS);u.verify(PROJECT_ROOT,HANDOFF_ONLY_INPUTS)
for rel,h in EXPECTED_OUTPUTS.items():assert sha(PROJECT_ROOT/rel)==h
again=u.read_csv(PROJECT_ROOT/'data/processed/taipei_dataset_b2.csv',contract)
pd.testing.assert_frame_equal(again,frame,check_exact=True)
result={'status':'passed','dataset':STEM,'shape':list(frame.shape),'sha256':sha(PROJECT_ROOT/'data/processed/taipei_dataset_b2.csv'),'schema_sha256':sha(PROJECT_ROOT/'data/processed/taipei_dataset_b2_schema.json'),'independent_saved_file_read':True,'human_acceptance':'DE / DA / PM pending'}
temp=PROJECT_ROOT/'data/temp/unit25_validation';temp.mkdir(parents=True,exist_ok=True)
(temp/(STEM+'_documentation_summary.json')).write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
print(json.dumps(result,ensure_ascii=False,indent=2))

{
  "status": "passed",
  "dataset": "taipei_dataset_b2",
  "shape": [
    456,
    127
  ],
  "sha256": "1af7ce133e62a1188e8755a1e4e7bc55db8b89d639f089f1c75319c8a9ee0269",
  "schema_sha256": "d5857362444df6bea1f606330d0363d792cc1a7f56812c9bb4917ce1b5e3a102",
  "independent_saved_file_read": true,
  "human_acceptance": "DE / DA / PM pending"
}
